In [65]:
import pandas as pd

In [66]:
filepath = "nrc_data_csv/estonian_nrc.csv"
df = pd.read_csv(filepath)
df.head(12)

,word,anger,disgust,fear,joy,sadness,surprise
0,hämmeldunud,0,0,0,0,0,0
1,aabits,0,0,0,0,0,0
2,hülgama,0,0,1,0,1,0
3,mahajäetud,1,0,1,0,1,0
4,hülgamine,1,0,1,0,1,1
5,raugema,0,0,0,0,0,0
6,vähendamine,0,0,0,0,0,0
7,abba,0,0,0,0,0,0
8,abt,0,0,0,0,0,0
9,lühendada,0,0,0,0,0,0


In [67]:
from datasets import load_dataset
import pandas as pd

target_emotions = ["anger", "disgust", "fear", "joy", "sadness", "surprise"]
random_state = 42

# Load German evaluation data

comb_df = pd.concat(
    map(pd.read_csv, ["xed_estonian.csv"]),
    ignore_index=True
)

comb_df["text"] = comb_df["text"].astype(str)
comb_df["emotion"] = comb_df["emotion"].astype(str)

print("Original German rows:", len(comb_df))
print("\nOriginal German class distribution:")
print(comb_df["emotion"].value_counts())


# Balance German data
target_n = comb_df["emotion"].value_counts().min()

balanced_parts = []
for emotion, group in comb_df.groupby("emotion"):
    balanced_parts.append(
        group.sample(n=target_n, random_state=random_state)
    )

comb_df = (
    pd.concat(balanced_parts, ignore_index=True)
    .sample(frac=1, random_state=random_state)
    .reset_index(drop=True)
)

print("\nBalanced German rows:", len(comb_df))
print("\nBalanced German class distribution:")
print(comb_df["emotion"].value_counts())

Original German rows: 674

Original German class distribution:
emotion
anger       205
fear        106
joy         105
surprise    102
sadness     101
disgust      55
Name: count, dtype: int64

Balanced German rows: 330

Balanced German class distribution:
emotion
anger       55
fear        55
disgust     55
surprise    55
joy         55
sadness     55
Name: count, dtype: int64


In [68]:
import stanza

stanza.download("et")

print(df)

2026-05-14 13:40:55 INFO: Downloaded file to /Users/arsenimuravjov/Library/Caches/stanza/1.11.0/resources/resources.json
2026-05-14 13:40:55 INFO: Downloading default packages for language: et (Estonian) ...
2026-05-14 13:40:56 INFO: File exists: /Users/arsenimuravjov/Library/Caches/stanza/1.11.0/resources/et/default.zip
2026-05-14 13:40:57 INFO: Finished downloading models and saved to /Users/arsenimuravjov/Library/Caches/stanza/1.11.0/resources


               word  anger  disgust  fear  joy  sadness  surprise
0       hämmeldunud      0        0     0    0        0         0
1            aabits      0        0     0    0        0         0
2           hülgama      0        0     1    0        1         0
3        mahajäetud      1        0     1    0        1         0
4         hülgamine      1        0     1    0        1         1
...             ...    ...      ...   ...  ...      ...       ...
14149        tsooni      0        0     0    0        0         0
14150      loomaaed      0        0     0    0        0         0
14151  zooloogiline      0        0     0    0        0         0
14152     zooloogia      0        0     0    0        0         0
14153         suumi      0        0     0    0        0         0

[14154 rows x 7 columns]


In [69]:
import stanza

nlp = stanza.Pipeline(
    lang="et",
    processors="tokenize,pos,lemma",
    use_gpu=True  # optional
)

doc = nlp("tere, minu nimi on sinu jalga")

for sent in doc.sentences:
    for word in sent.words:
        print(word.text, "->", word.lemma)

2026-05-14 13:40:58 INFO: Checking for updates to resources.json in case models have been updated.  Note: this behavior can be turned off with download_method=None or download_method=DownloadMethod.REUSE_RESOURCES
2026-05-14 13:40:58 INFO: Downloaded file to /Users/arsenimuravjov/Library/Caches/stanza/1.11.0/resources/resources.json
2026-05-14 13:40:58 WARNING: Language et package default expects mwt, which has been added
2026-05-14 13:40:58 INFO: Loading these models for language: et (Estonian):
| Processor | Package      |
----------------------------
| tokenize  | edt          |
| mwt       | edt          |
| pos       | edt_nocharlm |
| lemma     | edt_nocharlm |

2026-05-14 13:40:58 WARNING: GPU requested, but is not available!
2026-05-14 13:40:58 INFO: Using device: cpu
2026-05-14 13:40:58 INFO: Loading: tokenize
2026-05-14 13:40:58 INFO: Loading: mwt
2026-05-14 13:40:58 INFO: Loading: pos
2026-05-14 13:40:59 INFO: Loading: lemma
2026-05-14 13:41:00 INFO: Done loading processors!

tere -> tere
, -> ,
minu -> mina
nimi -> nimi
on -> olema
sinu -> sina
jalga -> jalg


In [70]:
import random

random.seed(42)

df["word"] = df["word"].astype(str).str.lower().str.strip()

df = (
    df.groupby("word", as_index=False)[target_emotions]
    .max()
)

lexicon = df.set_index("word")[target_emotions].to_dict(orient="index")

emotion_frequency = {}

for word, scores in lexicon.items():
    active_count = sum(scores[emotion] for emotion in target_emotions)

    if active_count > 0:
        emotion_frequency[word] = 1 / active_count
    else:
        emotion_frequency[word] = 0

predictions = []

for text in comb_df["text"]:
    count = {emotion: 0 for emotion in target_emotions}

    doc = nlp(text)

    for sent in doc.sentences:
        for word in sent.words:
            candidates = [
                word.lemma.lower().strip() if word.lemma else "",
                word.text.lower().strip()
            ]

            for candidate in candidates:
                if candidate in lexicon:
                    word_weight = emotion_frequency[candidate]

                    for emotion in target_emotions:
                        count[emotion] += lexicon[candidate][emotion] * word_weight
                    
                    break

    maxim = max(count.values())
    print(text)
    print(count)
    print()

    if maxim == 0:
        pred = "unknown"
    else:
        best_emotions = [emo for emo, score in count.items() if score == maxim]
        pred = random.choice(best_emotions)

    predictions.append(pred)

comb_df["prediction"] = predictions


Kardate seda hädist karu?
{'anger': 0.5, 'disgust': 0.0, 'fear': 0.5, 'joy': 0.0, 'sadness': 0.0, 'surprise': 0.0}

Me peaksime lahkuma enne kui on hilja.
{'anger': 0.0, 'disgust': 0.0, 'fear': 1.0, 'joy': 0.0, 'sadness': 2.0, 'surprise': 0.0}

Katke mind.
{'anger': 0, 'disgust': 0, 'fear': 0, 'joy': 0, 'sadness': 0, 'surprise': 0}

See on minu show.
{'anger': 0, 'disgust': 0, 'fear': 0, 'joy': 0, 'sadness': 0, 'surprise': 0}

Mölaka poeg.
{'anger': 0, 'disgust': 0, 'fear': 0, 'joy': 0, 'sadness': 0, 'surprise': 0}

Mulle see ei meeldi.
{'anger': 0, 'disgust': 0, 'fear': 0, 'joy': 0, 'sadness': 0, 'surprise': 0}

- Sa kasutasid tema pead peksukotina.
{'anger': 0, 'disgust': 0, 'fear': 0, 'joy': 0, 'sadness': 0, 'surprise': 0}

Kena sisenemine.
{'anger': 0, 'disgust': 0, 'fear': 0, 'joy': 0, 'sadness': 0, 'surprise': 0}

Ma poleks kunagi arvanud...
{'anger': 0, 'disgust': 0, 'fear': 0, 'joy': 0, 'sadness': 0, 'surprise': 0}

Me teeme su vanaisale suure üllatuse.
{'anger': 0.0, 'disgust'

In [71]:
for text in comb_df["text"].head(5):
    doc = nlp(text)
    print("\nTEXT:", text)

    for sent in doc.sentences:
        for word in sent.words:
            lemma = word.lemma.lower().strip() if word.lemma else word.text.lower().strip()
            print(word.text, "->", lemma, "MATCH" if lemma in lexicon else "")


TEXT: Kardate seda hädist karu?
Kardate -> kartma 
seda -> see 
hädist -> hädist 
karu -> karu MATCH
? -> ? 

TEXT: Me peaksime lahkuma enne kui on hilja.
Me -> mina 
peaksime -> pidama 
lahkuma -> lahkuma MATCH
enne -> enne MATCH
kui -> kui 
on -> olema 
hilja -> hilja MATCH
. -> . 

TEXT: Katke mind.
Katke -> katma 
mind -> mina 
. -> . 

TEXT: See on minu show.
See -> see 
on -> olema 
minu -> mina 
show -> show 
. -> . 

TEXT: Mölaka poeg.
Mölaka -> mölakas 
poeg -> poeg MATCH
. -> . 


In [72]:
from sklearn.metrics import accuracy_score, balanced_accuracy_score, f1_score, classification_report, confusion_matrix
import pandas as pd
import numpy as np

target_emotions = ["anger", "disgust", "fear", "joy", "sadness", "surprise"]

def evaluate_lexicon_predictions(df, gold_col="emotion", pred_col="prediction"):
    """
    Evaluates lexicon predictions.
    Unknown predictions are counted as wrong in the main evaluation.
    Also reports coverage and covered-only performance.
    """

    eval_df = df.copy()
    eval_df[gold_col] = eval_df[gold_col].astype(str)
    eval_df[pred_col] = eval_df[pred_col].astype(str)

    # -------------------------
    # Main evaluation: unknown = wrong
    # -------------------------
    y_true = eval_df[gold_col]
    y_pred = eval_df[pred_col]

    print("=== Main evaluation: unknown counted as wrong ===")
    print("Accuracy:", round(accuracy_score(y_true, y_pred), 4))
    print("Balanced accuracy:", round(balanced_accuracy_score(y_true, y_pred), 4))
    print("Macro F1:", round(f1_score(y_true, y_pred, labels=target_emotions, average="macro"), 4))
    print("Weighted F1:", round(f1_score(y_true, y_pred, labels=target_emotions, average="weighted"), 4))
    print()

    print(classification_report(
        y_true,
        y_pred,
        labels=target_emotions,
        digits=3,
        zero_division=0
    ))

    cm = confusion_matrix(y_true, y_pred, labels=target_emotions)
    cm_df = pd.DataFrame(cm, index=target_emotions, columns=target_emotions)

    print("Confusion matrix:")
    print(cm_df)
    print()

    # -------------------------
    # Coverage
    # -------------------------
    covered_mask = eval_df[pred_col] != "unknown"
    coverage = covered_mask.mean()

    print("=== Coverage ===")
    print("Coverage:", round(coverage, 4))
    print("Covered examples:", covered_mask.sum())
    print("Total examples:", len(eval_df))
    print()

    # -------------------------
    # Covered-only evaluation
    # -------------------------
    covered_df = eval_df[covered_mask].copy()

    if len(covered_df) > 0:
        print("=== Covered-only evaluation ===")
        print("Accuracy:", round(accuracy_score(covered_df[gold_col], covered_df[pred_col]), 4))
        print("Balanced accuracy:", round(balanced_accuracy_score(covered_df[gold_col], covered_df[pred_col]), 4))
        print("Macro F1:", round(f1_score(
            covered_df[gold_col],
            covered_df[pred_col],
            labels=target_emotions,
            average="macro",
            zero_division=0
        ), 4))
        print()

        print(classification_report(
            covered_df[gold_col],
            covered_df[pred_col],
            labels=target_emotions,
            digits=3,
            zero_division=0
        ))
    else:
        print("No covered examples.")

    results = {
        "accuracy_unknown_wrong": accuracy_score(y_true, y_pred),
        "balanced_accuracy_unknown_wrong": balanced_accuracy_score(y_true, y_pred),
        "macro_f1_unknown_wrong": f1_score(y_true, y_pred, labels=target_emotions, average="macro", zero_division=0),
        "weighted_f1_unknown_wrong": f1_score(y_true, y_pred, labels=target_emotions, average="weighted", zero_division=0),
        "coverage": coverage,
    }

    if len(covered_df) > 0:
        results.update({
            "accuracy_covered_only": accuracy_score(covered_df[gold_col], covered_df[pred_col]),
            "balanced_accuracy_covered_only": balanced_accuracy_score(covered_df[gold_col], covered_df[pred_col]),
            "macro_f1_covered_only": f1_score(
                covered_df[gold_col],
                covered_df[pred_col],
                labels=target_emotions,
                average="macro",
                zero_division=0
            ),
        })

    return results

In [73]:
results = evaluate_lexicon_predictions(
    comb_df,
    gold_col="emotion",
    pred_col="prediction"
)

=== Main evaluation: unknown counted as wrong ===
Accuracy: 0.0636
Balanced accuracy: 0.0636
Macro F1: 0.0945
Weighted F1: 0.0945

              precision    recall  f1-score   support

       anger      0.077     0.018     0.029        55
     disgust      0.333     0.073     0.119        55
        fear      0.143     0.073     0.096        55
         joy      0.292     0.127     0.177        55
     sadness      0.167     0.055     0.082        55
    surprise      0.222     0.036     0.062        55

   micro avg      0.202     0.064     0.097       330
   macro avg      0.206     0.064     0.095       330
weighted avg      0.206     0.064     0.095       330

Confusion matrix:
          anger  disgust  fear  joy  sadness  surprise
anger         1        3     6    0        2         3
disgust       2        4     6    2        4         0
fear          5        1     4    4        4         1
joy           0        0     3    7        1         2
sadness       3        1     6   

/Users/arsenimuravjov/miniforge3/envs/xed-llm/lib/python3.10/site-packages/sklearn/metrics/_classification.py:2801: UserWarning: y_pred contains classes not in y_true
  warnings.warn("y_pred contains classes not in y_true")
/Users/arsenimuravjov/miniforge3/envs/xed-llm/lib/python3.10/site-packages/sklearn/metrics/_classification.py:2801: UserWarning: y_pred contains classes not in y_true
  warnings.warn("y_pred contains classes not in y_true")
